In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

df = pd.read_csv('KO_weekly_return_volatility.csv')
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')

train = df[df['Year'].isin([2021, 2022, 2023])].reset_index(drop=True)
test = df[df['Year'].isin([2024, 2025])].reset_index(drop=True)
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

print(X_train.shape)

svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train, Y_train)
preds_linear = svm_linear.predict(X_test)

(156, 2)


## Q1 : Implement linear SVM and compute accuracy for testing set


In [2]:
linear_acc = round(svm_linear.score(X_test, Y_test) * 100, 2)
print(f'Linear SVM accuracy: {linear_acc}%')

Linear SVM accuracy: 68.87%


## Q2 : Confusion matrix for testing set


In [3]:
cm = confusion_matrix(Y_test, preds_linear, labels=['YES', 'NO'])
print('Confusion Matrix (Linear SVM):')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm[0][0]}           {cm[0][1]}')
print(f'Actual NO   {cm[1][0]}           {cm[1][1]}')

Confusion Matrix (Linear SVM):
  Predicted YES  Predicted NO
Actual YES  0           33
Actual NO   0           73


## Q3 : True positive rate and true negative rate for testing set


In [4]:
TP, FN, FP, TN = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
print(f'TPR (sensitivity): {round(TP / (TP + FN) * 100, 2)}%')
print(f'TNR (specificity): {round(TN / (TN + FP) * 100, 2)}%')

TPR (sensitivity): 0.0%
TNR (specificity): 100.0%


## Q4 : Gaussian SVM accuracy, is it better than linear?


In [5]:
svm_gaussian = SVC(kernel='rbf')
svm_gaussian.fit(X_train, Y_train)
preds_gaussian = svm_gaussian.predict(X_test)
gaussian_acc = round(svm_gaussian.score(X_test, Y_test) * 100, 2)
print(f'Gaussian SVM accuracy: {gaussian_acc}%')
print(f'{"Gaussian is better than Linear" if gaussian_acc > linear_acc else "Linear is better or equal to Gaussian"}')

Gaussian SVM accuracy: 68.87%
Linear is better or equal to Gaussian


## Q5 : Polynomial SVM (degree=2) accuracy, is it better than linear?


In [6]:
svm_poly = SVC(kernel='poly', degree=2)
svm_poly.fit(X_train, Y_train)
preds_poly = svm_poly.predict(X_test)
poly_acc = round(svm_poly.score(X_test, Y_test) * 100, 2)
print(f'Polynomial SVM (degree=2) accuracy: {poly_acc}%')
print(f'{"Polynomial is better than Linear" if poly_acc > linear_acc else "Linear is better or equal to Polynomial"}')

Polynomial SVM (degree=2) accuracy: 68.87%
Linear is better or equal to Polynomial


## Q6 : Trading strategy (linear SVM) vs buy-and-hold

In [7]:
test['linear_prediction'] = preds_linear

initial_balance = 100
linear_balance = initial_balance
position = None

for week_idx in range(len(test)):
    current_week = test.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                linear_balance = linear_balance * (1 + row['Return'] / 100)

    if week_idx < len(test) - 1:
        next_prediction = test.iloc[week_idx + 1]['linear_prediction']
        position = 'invested' if next_prediction == 'YES' else None

bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'Linear SVM Strategy Final Balance: ${linear_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
winner = 'Linear SVM Strategy' if linear_balance > bh_balance else 'Buy-and-Hold'
print(f'{winner} results in a larger amount at the end of the testing period')

Linear SVM Strategy Final Balance: $100.00
Buy and Hold Final Balance: $126.14
Buy-and-Hold results in a larger amount at the end of the testing period
